# Strava Tell Me More 🚴‍♂️

Generate engaging historical stories about your real cycling routes using Strava API and OpenAI.

## Features
- 🚴‍♂️ Real Strava API integration
- 🤖 AI-powered storytelling with multiple themes
- 🔒 Secure environment variable configuration
- 📊 Smart fallback to mock data when needed

In [32]:
# Import required libraries
import pandas as pd
import os
import openai
from typing import List, Dict, Optional
import json
from datetime import datetime
import textwrap
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
print("📦 Libraries imported successfully!")

📦 Libraries imported successfully!


In [33]:
# Configuration - API keys from environment variables
openai_key = os.getenv('OPENAI_API_KEY')
strava_client_id = os.getenv('STRAVA_CLIENT_ID')
strava_client_secret = os.getenv('STRAVA_CLIENT_SECRET')
strava_access_token = os.getenv('STRAVA_ACCESS_TOKEN')

if not openai_key:
    raise ValueError("OPENAI_API_KEY environment variable not set. Please check your .env file.")

chat_model = "gpt-3.5-turbo"
print("🔑 Configuration loaded from environment variables")

🔑 Configuration loaded from environment variables


In [34]:
# Strava API Setup
try:
    from stravalib import Client
    from stravalib.exc import Fault as StravaFault
    
    # Initialize Strava client
    strava_client = None
    if strava_access_token:
        strava_client = Client(access_token=strava_access_token)
        print("✅ Strava API client initialized successfully")
    else:
        print("⚠️  Strava access token not found. Will use mock data.")
        
except ImportError:
    print("⚠️  stravalib not installed. Install with: pip install stravalib")
    strava_client = None
except Exception as e:
    print(f"❌ Error initializing Strava client: {e}")
    strava_client = None

✅ Strava API client initialized successfully


In [35]:
class StravaAPI:
    """Handles Strava API interactions for fetching cycling activities and route data."""
    
    def __init__(self, client=None):
        self.client = client
        self.mock_locations = ["Segbroek", "Kraayenstein", "Kwintsheul", "Schipluiden", 
                              "Berkel en Rodenrijs", "Pijnacker", "Leidschendam", "Marlot", "Haagse Bos"]
    
    def get_recent_activities(self, limit=10):
        """Fetch recent cycling activities from Strava."""
        if not self.client:
            print("🔄 No Strava client available, using mock data")
            return self._get_mock_activities()
            
        try:
            activities = list(self.client.get_activities(limit=limit))
            cycling_activities = [a for a in activities if a.type.lower() in ['ride', 'cycling']]
            
            if not cycling_activities:
                print("🔄 No cycling activities found, using mock data")
                return self._get_mock_activities()
                
            print(f"✅ Found {len(cycling_activities)} cycling activities")
            return cycling_activities
            
        except Exception as e:
            print(f"❌ Error fetching activities: {e}")
            print("🔄 Falling back to mock data")
            return self._get_mock_activities()
    
    def get_activity_details(self, activity_id):
        """Fetch detailed activity data including GPS coordinates."""
        if not self.client:
            return None
            
        try:
            activity = self.client.get_activity(activity_id)
            return activity
        except Exception as e:
            print(f"❌ Error fetching activity details for {activity_id}: {e}")
            return None
    
    def extract_route_coordinates(self, activity):
        """Extract GPS coordinates from activity streams."""
        if not self.client or not activity:
            return []
            
        try:
            # Get the activity's streams (GPS data)
            streams = self.client.get_activity_streams(
                activity.id, 
                types=['latlng'], 
                resolution='medium'
            )
            
            if 'latlng' in streams:
                coordinates = streams['latlng'].data
                # Sample coordinates to reduce processing (every 10th point)
                sampled_coords = coordinates[::10] if len(coordinates) > 20 else coordinates
                return sampled_coords
            else:
                print("⚠️  No GPS coordinates found for this activity")
                return []
                
        except Exception as e:
            print(f"❌ Error extracting coordinates: {e}")
            return []
    
    def _get_mock_activities(self):
        """Return mock activity data when Strava API is unavailable."""
        class MockActivity:
            def __init__(self, name, id):
                self.name = name
                self.id = id
                self.type = "Ride"
                
        return [
            MockActivity("Morning Ride through Dutch Countryside", 1),
            MockActivity("Historic Villages Tour", 2)
        ]
    
    def get_mock_locations(self):
        """Return mock Dutch cycling locations."""
        return self.mock_locations

print("✅ StravaAPI class defined")

✅ StravaAPI class defined


In [36]:
class RouteProcessor:
    """Processes GPS coordinates and extracts location information."""
    
    def __init__(self):
        self.netherlands_cities = [
            "Amsterdam", "Rotterdam", "The Hague", "Utrecht", "Eindhoven", "Tilburg",
            "Groningen", "Almere", "Breda", "Nijmegen", "Enschede", "Haarlem",
            "Arnhem", "Amersfoort", "Zaanstad", "Apeldoorn", "s-Hertogenbosch",
            "Hoofddorp", "Maastricht", "Leiden", "Dordrecht", "Zoetermeer",
            "Zwolle", "Deventer", "Delft", "Alkmaar", "Leeuwarden", "Venlo"
        ]
    
    def coordinates_to_locations(self, coordinates):
        """Convert GPS coordinates to location names."""
        if not coordinates:
            return []
            
        locations = []
        
        # Simple approach: map coordinate regions to known Dutch locations
        # This is a simplified version - in production you'd use reverse geocoding
        for lat, lng in coordinates:
            location = self._approximate_location(lat, lng)
            if location and location not in locations:
                locations.append(location)
        
        return locations[:10]  # Limit to 10 locations
    
    def _approximate_location(self, lat, lng):
        """Approximate location based on coordinates (simplified approach)."""
        # Netherlands is roughly between 50.75-53.7°N and 3.2-7.2°E
        if not (50.5 <= lat <= 53.8 and 3.0 <= lng <= 7.5):
            return None
            
        # Simple grid-based approach for demonstration
        # In production, use actual reverse geocoding API
        lat_regions = {
            (52.0, 52.2): ["The Hague", "Delft", "Leidschendam"],
            (52.2, 52.4): ["Leiden", "Haarlem", "Amsterdam"],
            (52.4, 52.6): ["Amsterdam", "Zaanstad", "Alkmaar"],
            (51.8, 52.0): ["Rotterdam", "Dordrecht", "Breda"],
            (51.4, 51.8): ["Tilburg", "s-Hertogenbosch", "Eindhoven"]
        }
        
        for (min_lat, max_lat), cities in lat_regions.items():
            if min_lat <= lat <= max_lat:
                # Return a random city from the region
                import random
                return random.choice(cities)
        
        # Fallback to a random Dutch city
        import random
        return random.choice(self.netherlands_cities)
    
    def process_route_data(self, strava_api, activity_id=None):
        """Process route data from Strava or return mock data."""
        if not strava_api.client or not activity_id:
            print("🔄 Using mock location data")
            return strava_api.get_mock_locations()
        
        try:
            # Get activity details
            activity = strava_api.get_activity_details(activity_id)
            if not activity:
                return strava_api.get_mock_locations()
            
            # Extract coordinates
            coordinates = strava_api.extract_route_coordinates(activity)
            if not coordinates:
                return strava_api.get_mock_locations()
            
            # Convert to location names
            locations = self.coordinates_to_locations(coordinates)
            if not locations:
                return strava_api.get_mock_locations()
            
            print(f"✅ Processed {len(locations)} locations from route data")
            return locations
            
        except Exception as e:
            print(f"❌ Error processing route data: {e}")
            return strava_api.get_mock_locations()
    
    def create_route_dataframe(self, locations):
        """Create a DataFrame from location list."""
        return pd.DataFrame({
            "country": ["The Netherlands"] * len(locations),
            "towns": locations
        })

print("✅ RouteProcessor class defined")

✅ RouteProcessor class defined


In [37]:
class StoryGenerator:
    """Generates AI-powered stories about cycling routes using OpenAI."""
    
    def __init__(self, api_key, model="gpt-3.5-turbo"):
        self.model = model
        openai.api_key = api_key
        
    def generate_city_summary(self, city_name):
        """Generate a summary for a single city."""
        prompt = f"Provide a brief historical summary of {city_name}, focusing on origins, importance, major battles or wars, and famous people. Keep it concise and engaging for a cyclist who has visited this place."
        
        try:
            response = openai.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a knowledgeable guide summarizing city histories for cyclists, focusing on salient points and engaging storytelling."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Error generating summary for {city_name}: {str(e)}"
    
    def generate_route_story(self, locations, story_type="golden_age"):
        """Generate a narrative story about a cycling route through multiple locations."""
        if isinstance(locations, list):
            cities = ", ".join(locations)
        else:
            cities = locations
            
        story_prompts = {
            "golden_age": f"""Retell the cyclist's journey as if this was the Dutch Golden Age (17th century), starting and ending at the first location: {cities}.
            Weave a vivid narrative of the route, key historic events, and notable figures tied to these cities.
            Do not simply list the cities - tell the story as if the reader was actually there, experiencing the journey through cobblestone streets, past windmills, and through historic marketplaces.""",
            
            "modern": f"""Tell the story of a modern cyclist's journey through these Dutch locations: {cities}.
            Focus on the contrast between historical significance and contemporary life, mentioning how these places have evolved while maintaining their character.
            Include sensory details about the cycling experience - the feel of the bike paths, the sounds of the city, the architecture that spans centuries.""",
            
            "war_liberation": f"""Narrate a cycling journey through {cities} from the perspective of the liberation of the Netherlands in 1944-1945.
            Focus on the historical significance of these locations during WWII and their liberation, while weaving in the modern cycling experience through these historically significant places."""
        }
        
        prompt = story_prompts.get(story_type, story_prompts["golden_age"])
        
        try:
            response = openai.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a master storyteller who specializes in historical narratives and cycling adventures. Create immersive, engaging stories that transport the reader."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=600,
                temperature=0.3
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Error generating route story: {str(e)}"
    
    def save_story_to_file(self, story, filename, story_type="golden_age"):
        """Save generated story to a file with metadata."""
        story_data = {
            "story": story,
            "story_type": story_type,
            "generated_at": datetime.now().isoformat(),
            "model_used": self.model
        }
        
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(story_data, f, indent=2, ensure_ascii=False)
            print(f"📝 Story saved to {filename}")
        except Exception as e:
            print(f"❌ Error saving story: {e}")

print("✅ StoryGenerator class defined")

✅ StoryGenerator class defined


In [38]:
# Initialize all components
strava_api = StravaAPI(strava_client)
route_processor = RouteProcessor()
story_generator = StoryGenerator(openai_key, chat_model)

print("🚀 All components initialized successfully!")
print(f"📊 Strava client: {'✅ Connected' if strava_client else '❌ Not available (will use mock data)'}")
print(f"🤖 OpenAI API: {'✅ Ready' if openai_key else '❌ Not configured'}")

🚀 All components initialized successfully!
📊 Strava client: ✅ Connected
🤖 OpenAI API: ✅ Ready


In [39]:
# Fetch and process route data
print("🔍 Fetching recent cycling activities...")
activities = strava_api.get_recent_activities(limit=5)

# Use the first cycling activity if available
activity_id = None if not activities else getattr(activities[0], 'id', None)

# Process route data
print("🗺️  Processing route data...")
locations = route_processor.process_route_data(strava_api, activity_id)

# Create DataFrame
route_locations = route_processor.create_route_dataframe(locations)
print(f"📍 Found {len(locations)} locations for storytelling")

🔍 Fetching recent cycling activities...


❌ Error fetching activities: Unauthorized: Authorization Error: [{'resource': 'AccessToken', 'field': 'activity:read_permission', 'code': 'missing'}]
🔄 Falling back to mock data
🗺️  Processing route data...
❌ Error fetching activity details for 1: Not Found: Record Not Found: [{'resource': 'Activity', 'field': 'id', 'code': 'invalid'}]
📍 Found 9 locations for storytelling


In [40]:
# Display the route locations
print("📋 Route locations found:")
print(route_locations)

# Extract unique towns for storytelling
all_unique_towns = route_locations["towns"].unique()
joined_string = ", ".join(map(str, all_unique_towns))
print(f"\n🏘️  All unique towns: {textwrap.fill(joined_string, width=80)}")

📋 Route locations found:
           country                towns
0  The Netherlands             Segbroek
1  The Netherlands         Kraayenstein
2  The Netherlands           Kwintsheul
3  The Netherlands          Schipluiden
4  The Netherlands  Berkel en Rodenrijs
5  The Netherlands            Pijnacker
6  The Netherlands         Leidschendam
7  The Netherlands               Marlot
8  The Netherlands           Haagse Bos

🏘️  All unique towns: Segbroek, Kraayenstein, Kwintsheul, Schipluiden, Berkel en Rodenrijs, Pijnacker,
Leidschendam, Marlot, Haagse Bos


In [41]:
# Generate different types of stories
story_types = ["golden_age", "modern", "war_liberation"]

print("📚 Generating stories about your cycling route...\n")

for story_type in story_types:
    print(f"=== {story_type.replace('_', ' ').title()} Story ===")
    story = story_generator.generate_route_story(all_unique_towns, story_type)
    
    # Display the story with proper formatting
    print(textwrap.fill(story, width=80))
    
    # Save story to file
    filename = f"cycling_story_{story_type}.json"
    story_generator.save_story_to_file(story, filename, story_type)
    print(f"\n{'='*50}\n")

📚 Generating stories about your cycling route...

=== Golden Age Story ===
In the heart of the Dutch Golden Age, a young cyclist set out on a remarkable
journey that would weave through the picturesque landscapes and historic towns
of the Netherlands. The sun cast a warm glow over the quaint village of Segbroek
as our adventurer mounted their trusty steed, ready to embark on a voyage
through time.  Pedaling through the cobbled streets of Kraayenstein, the cyclist
passed by grand estates and lush gardens that whispered tales of opulence and
wealth from the Golden Age. The windmills stood tall against the azure sky,
their sails turning gracefully in the breeze, a symbol of the Dutch mastery of
engineering and innovation.  As the journey continued through Kwintsheul and
Schipluiden, the cyclist marveled at the bustling marketplaces where merchants
traded exotic goods from distant lands. The aroma of freshly baked bread and
fragrant spices filled the air, mingling with the sound of laughte

In [42]:
# Optional: Generate individual city summaries
print("🏛️  Want to learn more about individual cities? Uncomment and run the code below:")
print()

# Uncomment the following lines to generate individual city summaries:
# for city in all_unique_towns[:3]:  # Limit to first 3 cities to avoid API costs
#     print(f"=== {city} ===")
#     summary = story_generator.generate_city_summary(city)
#     print(textwrap.fill(summary, width=80))
#     print("\n" + "="*30 + "\n")

print("✅ Strava Tell Me More notebook complete!")
print("📊 Summary:")
print(f"   • Processed {len(all_unique_towns)} unique locations")
print(f"   • Generated {len(story_types)} different story types")
print(f"   • Data source: {'Real Strava activity' if strava_client and activity_id else 'Mock data'}")
print(f"   • Stories saved as JSON files in current directory")

🏛️  Want to learn more about individual cities? Uncomment and run the code below:

✅ Strava Tell Me More notebook complete!
📊 Summary:
   • Processed 9 unique locations
   • Generated 3 different story types
   • Data source: Real Strava activity
   • Stories saved as JSON files in current directory
